<a href="https://colab.research.google.com/github/kexincchen/NLP-Project/blob/main/GroupID__COMP90042_Project_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 2024 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 1.1 Import Libraries and Functions

In [1]:
import json
import pandas as pd
import numpy as np
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from gensim.models import KeyedVectors, Word2Vec
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import re
# from nltk.stem.snowball import SnowballStemmer
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

nltk.download('punkt')
stop_words = set(stopwords.words('english'))
stemmer =  PorterStemmer()

# use word2vec to create embeddings for claims and evidences
class word2vec:
	def __init__(self):
		self.word_vectors = None

	def train_model(self, texts, dimension=300):
		processed_sentences = [sent.split() for sent in texts]
		model = Word2Vec(
			sentences=processed_sentences,
			vector_size=dimension
		)
		word_vectors = model.wv
		word_vectors.save("word2vec.wordvectors")

	def load_word_vectors(self):
		try:
			self.word_vectors = KeyedVectors.load('word2vec.wordvectors', mmap='r')
			return self.word_vectors
		except Exception as e:
			print("No model exists. Please train the model first!")

	def get_embedding(self, sent_list, dimension=300):
		if not self.word_vectors:
			self.word_vectors = self.load_word_vectors()

		all_vec = np.zeros((len(sent_list), dimension))
		for i in range(len(sent_list)):
			num_words = len(sent_list[i])
			vec = np.zeros((dimension,))

			# find the word vector for each word in a sentence
			if num_words > 0:
				for word in sent_list[i].split():
					if word in self.word_vectors:
						vec += self.word_vectors[word]
					else:
						num_words -= 1
				# calculate the average word vector
				if num_words > 0:
					vec = np.divide(vec, num_words)
				all_vec[i] = vec
		return all_vec

def create_embedding(claims_text, evidence_text, embedding="doc2vec"):
	# convert the claim text into a vector using the specified embedding method
	if embedding == "word2vec":
		w2v = word2vec()
		claim_vec = w2v.get_embedding(claims_text)
		evidence_vec = w2v.get_embedding(evidence_text)
		return claim_vec, evidence_vec
	else:
		print("Please choose a embedding type from the following: tfidf, word2vec, doc2vec")


def create_embedding_matrix(vocab_size, word_vectors, word_index, embedding_dim):
    embedding_matrix = np.zeros((vocab_size, embedding_dim))
    for word, i in word_index.items():
        if word in word_vectors:
            embedding_vector = word_vectors[word]
            if embedding_vector is not None:
                embedding_matrix[i] = embedding_vector
            else:
                # If a word is not found, the row stays all zeros (or could be initialized randomly)
                embedding_matrix[i] = np.random.normal(scale=0.6, size=(embedding_dim, ))
    return embedding_matrix, embedding_dim

def preprocess_text(text, remove_stopwords=True):
	"""Preprocesses text by lowercasing, tokenizing, removing stopwords and stemming."""
	stemmer = PorterStemmer()
	tokens = word_tokenize(text.lower())
	if remove_stopwords:
		stop_words = set(stopwords.words('english'))
		filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum() and word not in stop_words]
	else:
		filtered_tokens = [stemmer.stem(word) for word in tokens if word.isalnum()]
	return ' '.join(filtered_tokens)

def preprocess_text_classification(text, stemmer=None, stop_words=None):
    # Lowercase the text
    text = text.lower()
    # Remove non-alphanumeric characters
    text = re.sub(r'\W+', ' ', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Remove stopwords
    if stop_words:
        text = ' '.join([word for word in text.split() if word not in stop_words])
    # Apply stemming
    if stemmer:
        text = ' '.join([stemmer.stem(word) for word in text.split()])
    return text

def load_data(filepath):
	"""Load JSON data."""
	with open(filepath, 'r') as file:
		data = json.load(file)
	return data

def save_to_json(filepath, data):
	""" Save a dictionary to a JSON file."""
	with open(filepath, 'w', encoding='utf-8') as f:
		json.dump(data, f, ensure_ascii=False, indent=4)

def preprocess_evidence(filepath, remove_stopwords=True):
	evidence_data = load_data(filepath)
	evidence_map = {eid: preprocess_text(text) for eid, text in evidence_data.items()}
	save_to_json(filepath, evidence_map)


def convert_to_df(data, labelled=True, remove_stopwords=True):
	data_for_dataframe = []
	for claim_id, claim_details in data.items():
		claim_text = preprocess_text(claim_details['claim_text'])
		if labelled:
			claim_label = claim_details['claim_label']
			eids = claim_details['evidences']
			data_for_dataframe.append({
					'claim_id': claim_id,
					'claim_text': claim_details['claim_text'],
					'claim_preprocessed': claim_text,
					'evidence': eids,
					'claim_label': claim_label
				})
		else:
			data_for_dataframe.append({
					'claim_id': claim_id,
					'claim_text': claim_details['claim_text'],
					'claim_preprocessed': claim_text,
				})

	df = pd.DataFrame(data_for_dataframe)
	return df

def text2seq(train_text, test_text, tokenizer_name, tokenizer):
    tokenizer.fit_on_texts(train_text + test_text)
    max_length = max(
        max([len(text.split()) for text in train_text]),
        max([len(text.split()) for text in test_text]),
    )
    print("Max length:", max_length)

    train_sequence = [
        tokenizer.text_to_sequences(text) for text in train_text
    ]  
    test_sequence = [
        tokenizer.text_to_sequences(text) for text in test_text
    ]  
    input_text_index = tokenizer.word_index

    # Optionally, save the tokenizer
    with open(tokenizer_name + ".pickle", "wb") as handle:
        pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)
    return train_sequence, input_text_index, test_sequence, max_length


def to_padding(train_df, test_df, tokenizer):
    # Initialize and fit the tokenizer on claim and evidence separately
    x_claims_seq, x_claims_word_index, y_claims_seq, max_claims_length = text2seq(
        train_df["claim"].tolist(),
        test_df["claim"].tolist(),
        "tokenizer_claims",
        tokenizer,
    )
    x_sents_seq, x_sents_word_index, y_sents_seq, max_sents_length = text2seq(
        train_df["evidence"].tolist(),
        test_df["evidence"].tolist(),
        "tokenizer_evidence",
        tokenizer,
    )

    # Padding sequences
    x_claims_data = pad_sequences(x_claims_seq, maxlen=max_claims_length)
    y_claims_data = pad_sequences(y_claims_seq, maxlen=max_claims_length)
    x_sents_data = pad_sequences(x_sents_seq, maxlen=max_sents_length)
    y_sents_data = pad_sequences(y_sents_seq, maxlen=max_sents_length)

    x_labels = np.array(train_df["label"].tolist())
    y_labels = np.array(test_df["label"].tolist())

    return (
        x_claims_data, x_sents_data, x_labels, x_claims_word_index,
        x_sents_word_index, y_claims_data, y_sents_data, y_labels
    )

def pad_sequences(sequences, maxlen=None):
    if maxlen is None:
        maxlen = max(len(seq) for seq in sequences)
    padded_sequences = np.zeros(
        (len(sequences), maxlen), dtype=int
    )  
    for i, sequence in enumerate(sequences):
        end = min(len(sequence), maxlen)
        padded_sequences[i, :end] = sequence[:end]
    return padded_sequences

class MyTokenizer:
    def __init__(self):
        self.word_index = {"<PAD>": 0, "<UNK>": 1}
        self.idx_to_token = {0: "<PAD>", 1: "<UNK>"}
        
    def fit_on_texts(self, texts):
        for text in texts:
            for word in text.split():
                if word not in self.word_index:
                    self.word_index[word] = len(self.word_index)
                    self.idx_to_token[self.word_index[word]] = word

    def text_to_sequences(self, text):
        return [self.word_index.get(word, self.word_index["<UNK>"]) for word in text.split()]
    
    def texts_to_sequences(self, texts):
        return [[self.word_index.get(word, self.word_index["<UNK>"]) for word in text.split()] for text in texts]


    def encode(self, text, max_length):
        tokens = self.text_to_sequences(text)
        if len(tokens) < max_length:
            tokens += [self.word_index["<PAD>"]] * (max_length - len(tokens))
        else:
            tokens = tokens[:max_length]
        return tokens
    
    def __call__(self, claims, evidences, max_length=512, return_tensors="pt"):
        self.fit_on_texts([claims, evidences])
        encoded_claims = self.encode(claims, max_length)
        encoded_evidences = self.encode(evidences, max_length)
        if return_tensors == "pt":
            return {
                "input_ids": torch.tensor([encoded_claims, encoded_evidences], dtype=torch.long)
            }
        return {"input_ids": [encoded_claims, encoded_evidences]}
    
class EvidenceDataset(Dataset):
    def __init__(self, claims, evidences, labels):
        self.claims = claims
        self.evidences = evidences
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        claim = torch.tensor(self.claims[idx], dtype=torch.long)
        evidence = torch.tensor(self.evidences[idx], dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.float)
        return {
            'claims': claim,
            'evidences': evidence,
            'labels': label
        }

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Clare\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


## 1.2 Data Preprocessing

In [2]:
# read data files
train_claims_data = load_data('data/train-claims.json')
evidence_data = load_data('data/evidence.json')
dev_claims_data = load_data('data/dev-claims.json')

# convert data files to dataframe
train_claims_df = convert_to_df(train_claims_data, labelled=True, remove_stopwords=True)
dev_claims_df = convert_to_df(dev_claims_data, labelled=False, remove_stopwords=True)

In [3]:
filtered_evidence = {}  # Dictionary to store preprocessed English evidence

for evi_id, evi_text in evidence_data.items():
    # Check if the text is in English
    words = nltk.word_tokenize(evi_text)
    english_words = [word for word in words if word.isalpha()]
    if len(english_words) / len(words) > 0.5:  # If more than half of the words are English words, consider it as English text
        # If it is in English, perform further preprocessing, such as removing stopwords
        english_text = ' '.join(english_words)
        filtered_evidence[evi_id] = english_text

# Output the number of filtered evidence
filtered_evidence_count = len(filtered_evidence)
save_to_json("filtered_evidence.json", filtered_evidence)

In [4]:
evidence_map = {eid: preprocess_text(text) for eid, text in evidence_data.items()}
save_to_json('preprocessed_evidence_map.json', evidence_map)

filtered_evidence = load_data("filtered_evidence.json")
filtered_evidence_map = {eid: preprocess_text(text) for eid, text in filtered_evidence.items()}
save_to_json('preprocessed_filtered_evidence_map.json', filtered_evidence_map)

In [5]:
evidence_map = load_data('preprocessed_evidence_map.json')
filtered_evidence_map = load_data("preprocessed_filtered_evidence_map.json")
evidence_df = pd.DataFrame(evidence_map.items(), columns=['id', 'evidence'])
filtered_evidence_df = pd.DataFrame(filtered_evidence_map.items(), columns=['id', 'evidence'])

train_claims_df['evidence_texts'] = train_claims_df['evidence'].apply(
	lambda x: [evidence_map[evidence_id] for evidence_id in x]
)

train_claims_text = train_claims_df['claim_preprocessed'].tolist()
dev_claims_text = dev_claims_df['claim_preprocessed'].tolist()
dev_claims_id = dev_claims_df['claim_id'].tolist()

evidence_id = list(evidence_map.keys())
evidence_texts  = list(evidence_map.values())

In [6]:
# Prepare for Classification Model Training
label_mapping = {
    "SUPPORTS": 1,
    "REFUTES": -1,
    "NOT_ENOUGH_INFO": 0
}
data_for_dataframe = []
for claim_id, claim_details in train_claims_data.items():
    claim_text = claim_details['claim_text']
    claim_label = claim_details['claim_label']
    eids = claim_details['evidences']
    if claim_label == "DISPUTED":
            continue
    for eid in eids:
        evidence_text = evidence_map[eid]
        
        data_for_dataframe.append({
                'claim_id': claim_id,
                'claim': preprocess_text_classification(claim_text, stemmer, stop_words),
                'evidence': evidence_text,
                'label': label_mapping[claim_label]
            })

# Create DataFrame
train_claims_df = pd.DataFrame(data_for_dataframe)

train_df, test_df = train_test_split(train_claims_df, test_size=0.2, random_state=42)

In [7]:
tokenizer = MyTokenizer()
x_claim, x_sents, x_labels, x_claims_word_index,  x_sents_word_index, y_claims_data, y_sents_data, y_labels = to_padding(train_df, test_df, tokenizer)

print ("x claim word index ", len(x_claims_word_index))
print ("x sent word index ", len(x_sents_word_index))

vocab_size_claims = len(x_claims_word_index) + 2  # +1 for padding, +1 for <UNK>
vocab_size_evidences = len(x_sents_word_index) + 2

# Assuming x_claim, y_claims_data, etc. are numpy arrays or lists of integers
train_dataset = EvidenceDataset(x_claim, x_sents, x_labels)
test_dataset = EvidenceDataset(y_claims_data, y_sents_data, y_labels)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=128, shuffle=False)


Max length: 35
Max length: 177
x claim word index  5804
x sent word index  5804


## 1.3 Create Embedding

In [8]:
w2v = word2vec()
w2v.train_model(train_claims_text + evidence_texts)

In [9]:
word_vectors = KeyedVectors.load('word2vec.wordvectors', mmap='r')

embedding_dim = 300  # dimension of word2vec vectors
(embed_matrix_claim, embed_dim_claim) = create_embedding_matrix(vocab_size_claims, word_vectors, x_claims_word_index, embedding_dim)
(embed_matrix_evidence, embed_dim_evidence) = create_embedding_matrix(vocab_size_evidences, word_vectors, x_sents_word_index, embedding_dim)

print ("embed_matrix_claim shape ", embed_matrix_claim.shape)
print ("embed_matrix_evidence shape ", embed_matrix_evidence.shape)

embed_matrix_claim shape  (5806, 300)
embed_matrix_evidence shape  (5806, 300)


# 2. Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 2.1 Evidence Retrieval

In [10]:
vectorizer = TfidfVectorizer()
vectorizer.fit(train_claims_text + evidence_texts)
evidence_vec = vectorizer.transform(evidence_texts)
dev_claims_vec = vectorizer.transform(dev_claims_text)
print(dev_claims_vec.shape)
print(evidence_vec.shape)

(154, 489491)
(1181638, 489491)


## 2.2 Claim Classification

In [11]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, embed_dim, num_heads):
        super(MultiHeadAttentionWrapper, self).__init__()
        self.multihead_attn = nn.MultiheadAttention(embed_dim=embed_dim, num_heads=num_heads)

    def forward(self, x):
        # Transpose batch and sequence dimensions:
        x = x.transpose(0, 1)
        attn_output, attn_output_weights = self.multihead_attn(x, x, x)
        return attn_output.transpose(0, 1), attn_output_weights

class EvidenceModel(nn.Module):
    def __init__(self, vocab_size_claims, vocab_size_evidences, embed_dim_claim, embed_dim_evidence):
        super(EvidenceModel, self).__init__()
        self.embedding_claims = nn.Embedding(vocab_size_claims, embed_dim_claim)
        self.lstm_claims_1 = nn.LSTM(embed_dim_claim, 64, batch_first=True, bidirectional=True)
        self.lstm_claims_2 = nn.LSTM(64*2, 32, batch_first=True) 
        self.attention_claims = MultiHeadAttentionWrapper(32, 2) 
        
        self.embedding_evidences = nn.Embedding(vocab_size_evidences, embed_dim_evidence)
        self.lstm_evidences_1 = nn.LSTM(embed_dim_evidence, 64, batch_first=True, bidirectional=True)
        self.lstm_evidences_2 = nn.LSTM(64*2, 32, batch_first=True) 
        self.attention_evidences = MultiHeadAttentionWrapper(32, 2)
        
        self.pool = nn.AdaptiveAvgPool1d(1)  
        self.dropout = nn.Dropout(0.5)
        self.fc1 = nn.Linear(32*2, 64)  
        self.fc2 = nn.Linear(64, 1)

    def forward(self, claims_input, evidences_input):
        embedded_claims = self.embedding_claims(claims_input)
        lstm_out_claims, _ = self.lstm_claims_1(embedded_claims)
        lstm_out_claims, _ = self.lstm_claims_2(lstm_out_claims)
        attentive_claims, _ = self.attention_claims(lstm_out_claims)

        embedded_evidences = self.embedding_evidences(evidences_input)
        lstm_out_evidences, _ = self.lstm_evidences_1(embedded_evidences)
        lstm_out_evidences, _ = self.lstm_evidences_2(lstm_out_evidences)
        attentive_evidences, _ = self.attention_evidences(lstm_out_evidences)

        # Pooling each output to reduce dimension to 1
        attentive_claims = self.pool(attentive_claims.transpose(1,2)).squeeze(-1)
        attentive_evidences = self.pool(attentive_evidences.transpose(1,2)).squeeze(-1)

        concatenated = torch.cat((attentive_claims, attentive_evidences), dim=1)
        concatenated = self.dropout(concatenated)
        concatenated = self.fc1(concatenated)
        output = self.fc2(concatenated)
        return torch.tanh(output)


In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = EvidenceModel(vocab_size_claims, vocab_size_evidences, embed_dim_claim, embed_dim_evidence).to(device)
criterion = nn.SmoothL1Loss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5) 


def train_model(model, train_loader, val_loader, epochs, device):
    model_path = 'checkpoint.pth'
    best_val_loss = float('inf')
    patience = 2
    trigger_times = 0

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            claims = batch['claims'].to(device)
            evidences = batch['evidences'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            outputs = model(claims, evidences).squeeze(1)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        avg_train_loss = total_loss / len(train_loader)
        val_loss = evaluate(model, val_loader, device)
        print(f'Epoch {epoch+1}, Train Loss: {avg_train_loss:.4f}, Val Loss: {val_loss:.4f}')

        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), model_path)
            trigger_times = 0
        else:
            trigger_times += 1
            if trigger_times >= patience:
                print(f'Early stopping at epoch {epoch+1}')
                break

def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for batch in loader:
            claims = batch['claims'].to(device)
            evidences = batch['evidences'].to(device)
            labels = batch['labels'].to(device)
            outputs = model(claims, evidences).squeeze(1)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
    return total_loss / len(loader)

def predict(model, data_loader, device):
    model.eval() 
    predictions = []
    with torch.no_grad():  
        for batch in data_loader:
            claims = batch['claims'].to(device)
            evidences = batch['evidences'].to(device)
            outputs = model(claims, evidences)
            predictions.append(outputs.cpu()) 
            
    # Concatenate the list of tensors into a single tensor
    predictions = torch.cat(predictions, dim=0)
    return predictions

In [13]:
# Train the model
epochs = 60
train_model(model, train_loader, test_loader, epochs, device)

def tag_label(output_value, threshold):
    if output_value > threshold:
        return 1
    if output_value < -threshold:
        return -1
    return 0
# Test loader setup like train_loader and val_loader
test_loss = evaluate(model, test_loader, device)
print("Test loss", test_loss)
threshold = 0.2
# Prediction and performance metrics
y_pred = []
y_true = []
model.eval()
with torch.no_grad():
    for batch in test_loader:
        claims = batch['claims'].to(device)
        evidences = batch['evidences'].to(device)
        labels = batch['labels'].cpu().numpy()
        outputs = model(claims, evidences).cpu().numpy()[:, 0]  # Adjust shape handling as necessary
        predicted_labels = [tag_label(output, threshold) for output in outputs]
        y_pred.extend(predicted_labels)
        y_true.extend(labels)

accuracy = accuracy_score(y_true, y_pred)
precision, recall, fscore, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F-score:", fscore)

Epoch 1, Train Loss: 0.2184, Val Loss: 0.2063
Epoch 2, Train Loss: 0.2142, Val Loss: 0.2060
Epoch 3, Train Loss: 0.2076, Val Loss: 0.1908
Epoch 4, Train Loss: 0.1417, Val Loss: 0.1202
Epoch 5, Train Loss: 0.0733, Val Loss: 0.0762
Epoch 6, Train Loss: 0.0366, Val Loss: 0.0622
Epoch 7, Train Loss: 0.0235, Val Loss: 0.0548
Epoch 8, Train Loss: 0.0175, Val Loss: 0.0503
Epoch 9, Train Loss: 0.0132, Val Loss: 0.0472
Epoch 10, Train Loss: 0.0103, Val Loss: 0.0480
Epoch 11, Train Loss: 0.0081, Val Loss: 0.0467
Epoch 12, Train Loss: 0.0072, Val Loss: 0.0475
Epoch 13, Train Loss: 0.0062, Val Loss: 0.0462
Epoch 14, Train Loss: 0.0058, Val Loss: 0.0441
Epoch 15, Train Loss: 0.0048, Val Loss: 0.0437
Epoch 16, Train Loss: 0.0039, Val Loss: 0.0426
Epoch 17, Train Loss: 0.0031, Val Loss: 0.0437
Epoch 18, Train Loss: 0.0028, Val Loss: 0.0432
Early stopping at epoch 18
Test loss 0.04322600644081831
Accuracy: 0.9463806970509383
Precision: 0.9177872437867611
Recall: 0.9334026203073823
F-score: 0.924433305

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## 3.1 Evidence Retrieval

In [14]:
def top_k_evidence(claims_id, claims_emb, evidence_emb, evidence_df, k=3):
	"""
	input:
		claims_id: list of claims' id, (N_c, )
		claims_emb: matrix of claims' embedding, (N_c, d)
		evidence_emb: matrix of evidences' embedding, (N_e, d)
		k: number of evidences selected for each claim

	output:
		top_evidence_id: dictionary that contains claims_id and their corresponding evidences, {claim_id1: [eid1, eid2, ...], claim_id2: []}
	"""
	sim = cosine_similarity(claims_emb, evidence_emb)

	# get top k evidences with highest similarity score with the claim
	data = np.zeros((sim.shape[0], k))
	top_evidence_id = {}
	for i in range(sim.shape[0]):
		data[i] = np.argpartition(sim[i], -k)[-k:]
		top_evidence_id[claims_id[i]] = [evidence_df.iloc[int(ind)]['id'] for ind in data[i]]
	return top_evidence_id

top_evidence_id = top_k_evidence(dev_claims_id, dev_claims_vec, evidence_vec, evidence_df, k=3)

with open('data/dev-claims.json', 'r') as input_file:
    test_out_temp = json.load(input_file)

for claim_id, _ in test_out_temp.items():
	test_out_temp[claim_id]["evidences"] = top_evidence_id[claim_id]

with open("dev_predict.json", "w") as outfile:
    json.dump(test_out_temp, outfile)

## 3.2 Claim Classification

In [15]:
dev_predicted = load_data("dev_predict.json")

data_for_dataframe = []
label_only = []
for claim_id, claim_details in dev_predicted.items():
    claim_text = preprocess_text_classification(claim_details['claim_text'], stemmer, stop_words)
    eids = claim_details['evidences']
    label_only.append({
        'claim_id': claim_id,
        'claim_label': claim_details['claim_label']
    })
    data_for_dataframe.append({
			'claim_id': claim_id,
            'claim': claim_text,
            'original_claim': claim_details['claim_text'],
            'evidence': eids
        })
    
# Create DataFrame
output_df = pd.DataFrame(data_for_dataframe)
label_df = pd.DataFrame(label_only)
data_for_dataframe = []
for claim_id, claim_details in dev_predicted.items():
    claim_text = preprocess_text_classification(claim_details['claim_text'], stemmer, stop_words)
    eids = claim_details['evidences']
    for eid in eids:
        evidence_text = evidence_map[eid]
        data_for_dataframe.append({
                'claim_id': claim_id,
                'claim': claim_text,
                'original_claim': claim_details['claim_text'],
                'evidence_id': eid,
                'evidence_text': evidence_text
            })
    
# Create DataFrame
dev_claims_df = pd.DataFrame(data_for_dataframe)

In [16]:
with open('tokenizer_claims.pickle', 'rb') as handle:
	claims_tokenizer = pickle.load(handle)
	
with open('tokenizer_evidence.pickle', 'rb') as handle:
	evidence_tokenizer = pickle.load(handle)

max_claims_length = 68
max_sents_length = 264

dev_sents_text = evidence_map.values()

# Tokenize the preprocessed text
dev_claims = claims_tokenizer.texts_to_sequences(dev_claims_df["claim"].tolist())
dev_sents = evidence_tokenizer.texts_to_sequences(dev_claims_df['evidence_text'].tolist())

# Padding sequences to ensure uniform input size
dev_claims = pad_sequences(dev_claims, maxlen=max_claims_length)
dev_sents = pad_sequences(dev_sents, maxlen=max_sents_length)

# Print shapes
print("dev claims shape:", dev_claims.shape)
print("dev sents shape:", dev_sents.shape)

dev claims shape: (462, 68)
dev sents shape: (462, 264)


In [17]:
# model.load_state_dict(torch.load('checkpoint.pth'))
model = model.to(device)  
model.eval()  

dev_claims = torch.tensor(dev_claims).to(device)  
dev_sents = torch.tensor(dev_sents).to(device)
claim_ids = dev_claims_df["claim_id"].tolist()

batch_size = 128  
all_predictions = []

# Ensure batching through the whole dataset
for i in range(0, dev_claims.shape[0], batch_size):
    claim_batch = dev_claims[i:i+batch_size]
    sent_batch = dev_sents[i:i+batch_size]

    with torch.no_grad():
        outputs = model(claim_batch, sent_batch).squeeze(1).cpu().numpy()

    # Collect batch predictions with claim_ids
    batch_predictions = list(zip(claim_ids[i:i+batch_size], outputs))
    all_predictions.extend(batch_predictions)

results_df = pd.DataFrame(all_predictions, columns=["claim_id", "prediction"])

def tag_label(prediction, tl=-0.2, th=0.2):
    if prediction > th:
        return 1
    if prediction < tl:
        return -1
    return 0

# Apply predictions to each row in the DataFrame
results_df['prediction'] = results_df.apply(lambda row: tag_label(row['prediction'], tl=-0.2, th=0.20), axis=1)

def determine_final_label(labels):
    if len(set(labels)) == 1:
        return labels.iloc[0]  # all labels are the same
    else:
        return "2"  # conflicting labels

final_labels = results_df.groupby('claim_id')['prediction'].agg(determine_final_label)

output_df = output_df.join(final_labels, on='claim_id')

def map_to_label(prediction):
    if prediction == 0:
        return "NOT_ENOUGH_INFO"
    if prediction == -1:
        return "REFUTES"
    if prediction == 1:
        return "SUPPORTS"
    return "DISPUTED"

output_df["claim_label"] = output_df["prediction"].apply(lambda x: map_to_label(x))
output_df


,claim_id,claim,original_claim,evidence,prediction,claim_label
0,claim-752,south australia expens electr world,[South Australia] has the most expensive elect...,"[evidence-786054, evidence-995049, evidence-80...",-1,REFUTES
1,claim-375,3 per cent total annual global emiss carbon di...,when 3 per cent of total annual global emissio...,"[evidence-1011788, evidence-1140012, evidence-...",0,NOT_ENOUGH_INFO
2,claim-1266,mean world 1c warmer pre industri time,This means that the world is now 1C warmer tha...,"[evidence-945233, evidence-694262, evidence-94...",-1,REFUTES
3,claim-871,happen zika may also good model second worri e...,"“As it happens, Zika may also be a good model ...","[evidence-548884, evidence-336512, evidence-47...",1,SUPPORTS
4,claim-2164,greenland lost tini fraction ice mass,Greenland has only lost a tiny fraction of its...,"[evidence-776422, evidence-1200633, evidence-9...",-1,REFUTES
...,...,...,...,...,...,...
149,claim-2400,suddenli label co2 pollut disservic ga play en...,"'To suddenly label CO2 as a ""pollutant"" is a d...","[evidence-808385, evidence-1085172, evidence-5...",-1,REFUTES
150,claim-204,natur orbit driven warm atmospher carbon dioxi...,"after a natural orbitally driven warming, atmo...","[evidence-584709, evidence-143580, evidence-61...",-1,REFUTES
151,claim-1426,mani world coral reef alreadi barren state con...,Many of the world’s coral reefs are already ba...,"[evidence-744921, evidence-580567, evidence-38...",1,SUPPORTS
152,claim-698,recent studi led lawrenc livermor nation labor...,A recent study led by Lawrence Livermore Natio...,"[evidence-281542, evidence-366949, evidence-62...",1,SUPPORTS


In [18]:
output_df.drop(columns=["claim", "prediction"], inplace=True)
output_df.rename(columns={"original_claim": "claim_text", "evidence": "evidences"}, inplace=True)
output_df.set_index('claim_id', inplace=True)
result = output_df.to_json(orient="index")
with open('dev-claims-output.json', 'w') as f:
    f.write(result)

## 3.3 Prediction on Dev-Claims

In [19]:
!python eval.py --predictions dev-claims-output.json --groundtruth data/dev-claims.json

Evidence Retrieval F-score (F)    = 0.08172541743970316
Claim Classification Accuracy (A) = 0.42207792207792205
Harmonic Mean of F and A          = 0.13693634665831314


## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*